# SEC Chunk Classifier Training
This notebook loads the JSONL data generated by the RAG pipeline and prepares it for multi-label SequenceClassification using ModernBERT.

In [ ]:
!aws s3 sync s3://sec-chunk-classifier-data/ ../training_data/data/

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EvalPrediction,
    EarlyStoppingCallback
)
from transformers.trainer_utils import get_last_checkpoint
from sklearn.metrics import f1_score, roc_auc_score, accuracy_score

sys.path.append(os.path.abspath('..'))
from config import SectionTypes, CHUNK_SIZES_CHARS
from training_data.storage import db

CATEGORIES = [k for k in SectionTypes.keys() if k != "OTHER"]
NUM_LABELS = len(CATEGORIES)
id2label = {i: label for i, label in enumerate(CATEGORIES)}
label2id = {label: i for i, label in enumerate(CATEGORIES)}

MODEL_ID = "jinaai/jina-embeddings-v2-small-en"

MAX_LENGTH = max(CHUNK_SIZES_CHARS)
MODEL_ABSOLUTE_LIMIT = 8192

if MAX_LENGTH > MODEL_ABSOLUTE_LIMIT:
    raise ValueError(f"Configured max chunk size ({MAX_LENGTH}) exceeds the model's absolute token limit ({MODEL_ABSOLUTE_LIMIT})!")

In [2]:
def load_and_prepare_data(limit_filings=None):
    """
    Incrementally read all .jsonl files using the storage Db class,
    convert labels to multi-hot encoding, balance the dataset, and return a HuggingFace Dataset.
    """
    raw_records = db.read_labeled_data(limit_filings=limit_filings)
    print(f"Loaded {len(raw_records)} raw chunks from Db.")
    
    positive_records = []
    negative_records = []
    
    for data in raw_records:
        text = data.get("text")
        if not text:
            continue
        
        labels_present = data.get("section_labels", [])
        multi_hot = []
        for cat in CATEGORIES:
            if cat in labels_present:
                multi_hot.append(1.0)
            else:
                multi_hot.append(0.0)
                
        record = {
            "text": text,
            "labels": multi_hot
        }
        
        if sum(multi_hot) > 0:
            positive_records.append(record)
        else:
            negative_records.append(record)
            
    # BALANCE THE DATASET
    # Keep all positives, but sample negatives to be 2x the number of positives (or whatever ratio)
    import random
    random.seed(42)
    target_negative_count = int(len(positive_records) * 1.5)
    
    if len(negative_records) > target_negative_count:
        negative_records = random.sample(negative_records, target_negative_count)
        
    prepared_records = positive_records + negative_records
    random.shuffle(prepared_records)
                
    df = pd.DataFrame(prepared_records)
    if df.empty:
        print("No data found!")
        return None
        
    print(f"Balanced Dataset: {len(positive_records)} positive chunks, {len(negative_records)} 'OTHER' chunks.")
        
    dataset = Dataset.from_pandas(df)
    dataset = dataset.train_test_split(test_size=0.2, seed=42)
    
    print(f"Train split: {len(dataset['train'])} examples")
    print(f"Test split: {len(dataset['test'])} examples")
    
    return dataset



In [3]:
dataset_dict = load_and_prepare_data(limit_filings=None)

Loaded 7204 raw chunks from Db.
Train split: 5763 examples
Test split: 1441 examples


In [4]:
tokenizer = AutoTokenizer.from_pretrained(trust_remote_code=True, MODEL_ID)

def tokenize_function(examples):
    tokenized = tokenizer(examples["text"])
    max_tokens_in_batch = max(len(ids) for ids in tokenized["input_ids"])
    
    if max_tokens_in_batch > MODEL_ABSOLUTE_LIMIT:
        raise ValueError(f"Found a chunk with {max_tokens_in_batch} tokens, exceeding model limit of {MODEL_ABSOLUTE_LIMIT}!")
        
    return tokenizer(
        examples["text"], 
        padding="longest", 
        truncation=True, 
        max_length=MAX_LENGTH
    )

tokenized_dataset = dataset_dict.map(tokenize_function, batched=True)

Map:   0%|          | 0/5763 [00:00<?, ? examples/s]

Map:   0%|          | 0/1441 [00:00<?, ? examples/s]

In [5]:
def compute_metrics(p: EvalPrediction):
    preds = p.predictions[0] if isinstance(p.predictions, tuple) else p.predictions
    
    probs = 1.0 / (1.0 + np.exp(-preds))
    
    y_pred = np.where(probs >= 0.5, 1, 0)
    y_true = p.label_ids
    
    try:
        roc_auc = roc_auc_score(y_true, probs, average='macro')
    except ValueError:
        roc_auc = 0.0
        
    f1_macro_average = f1_score(y_true=y_true, y_pred=y_pred, average='macro', zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    
    return {
        'roc_auc': roc_auc,
        'f1': f1_macro_average,
        'accuracy': accuracy
    }

In [6]:
model = AutoModelForSequenceClassification.from_pretrained(trust_remote_code=True, 
    MODEL_ID, 
    num_labels=NUM_LABELS,
    problem_type="multi_label_classification",
    id2label=id2label,
    label2id=label2id
)

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
training_args = TrainingArguments(
    output_dir="./stonky_sec_jinabert_small",
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=3,            # Keep only the last 3 checkpoints to save disk space
    learning_rate=2e-5,            # Keep LR low to avoid catastrophic forgetting
    per_device_train_batch_size=2, # Reduced to 2 to prevent MPS OOM crash
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4, # Accumulate gradients to keep effective batch size at 8
    num_train_epochs=10,           # Increased max epochs because Early Stopping will catch it  
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    fp16=True,                     # Enable Mixed Precision for speed on Mac/AWS
)

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

last_checkpoint = get_last_checkpoint("./stonky_sec_jinabert_small") if os.path.isdir("./stonky_sec_jinabert_small") else None
if last_checkpoint is not None:
    print(f"Resuming training from checkpoint: {last_checkpoint}")
else:
    print("Starting institutional multi-label fine-tuning...")
    
trainer.train(resume_from_checkpoint=last_checkpoint)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Resuming training from checkpoint: ./stonky_sec_modernbert/checkpoint-2163


/Users/marcinzarkowski/Code/stonky/sec-chunk-classifier/.venv/lib/python3.12/site-packages/torch/utils/data/dataloader.py:759: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


In [ ]:
trainer.save_model("./stonky_sec_jinabert_small_final")
tokenizer.save_pretrained("./stonky_sec_jinabert_small_final")
print("Best model saved to ./stonky_sec_jinabert_small_final")

In [ ]:
print("\n--- Evaluating Performance on the Test Set ---")
metrics = trainer.evaluate(eval_dataset=tokenized_dataset["test"])

print(f"\nTest Set Results:")
print(f"Macro F1 Score : {metrics['eval_f1']:.4f}")
print(f"ROC AUC Score  : {metrics['eval_roc_auc']:.4f}")
print(f"Hard Accuracy  : {metrics['eval_accuracy']:.4f}")

print("\nDetailed mapping for prediction sanity check:")
for k, v in id2label.items():
    print(f"Class {k}: {v}")

In [ ]:
import os
from transformers.trainer_utils import get_last_checkpoint

latest_ckpt = get_last_checkpoint("./stonky_sec_jinabert_small") if os.path.isdir("./stonky_sec_jinabert_small") else "./stonky_sec_jinabert_small_final"

if latest_ckpt and os.path.exists(latest_ckpt):
    print(f"\n--- Loading and Evaluating Latest Checkpoint: {latest_ckpt} ---")
    
    ckpt_model = AutoModelForSequenceClassification.from_pretrained(trust_remote_code=True, 
        latest_ckpt,
        num_labels=NUM_LABELS,
        problem_type="multi_label_classification",
        id2label=id2label,
        label2id=label2id
    )
    
    ckpt_trainer = Trainer(
        model=ckpt_model,
        args=training_args,
        eval_dataset=tokenized_dataset["test"],
        processing_class=tokenizer,
        compute_metrics=compute_metrics
    )
    
    ckpt_metrics = ckpt_trainer.evaluate()
    
    print(f"\nCheckpoint Test Set Results:")
    print(f"Macro F1 Score : {ckpt_metrics['eval_f1']:.4f}")
    print(f"ROC AUC Score  : {ckpt_metrics['eval_roc_auc']:.4f}")
    print(f"Hard Accuracy  : {ckpt_metrics['eval_accuracy']:.4f}")
else:
    print("No checkpoint found to evaluate! Run the training loop first.")